<p align="right"><a href="./C1_W3_Logistic_Regression.ipynb">English</a> | <strong>中文</strong></p>

# 逻辑回归（Logistic Regression）

在本练习中，你将实现逻辑回归，并把它应用到两个不同的数据集上。

# 大纲
- [ 1 - 库 Packages ](#1)
- [ 2 - 逻辑回归 Logistic Regression](#2)
  - [ 2.1 问题描述](#2.1)
  - [ 2.2 加载并可视化数据](#2.2)
  - [ 2.3  Sigmoid 函数](#2.3)
  - [ 2.4 逻辑回归的代价函数](#2.4)
  - [ 2.5 逻辑回归的梯度](#2.5)
  - [ 2.6 用梯度下降学习参数 ](#2.6)
  - [ 2.7 画出决策边界](#2.7)
  - [ 2.8 评估逻辑回归](#2.8)
- [ 3 - 正则化逻辑回归 Regularized Logistic Regression](#3)
  - [ 3.1 问题描述](#3.1)
  - [ 3.2 加载并可视化数据](#3.2)
  - [ 3.3 特征映射 Feature mapping](#3.3)
  - [ 3.4 正则化逻辑回归的代价函数](#3.4)
  - [ 3.5 正则化逻辑回归的梯度](#3.5)
  - [ 3.6 用梯度下降学习参数](#3.6)
  - [ 3.7 画出决策边界](#3.7)
  - [ 3.8 评估正则化逻辑回归模型](#3.8)

_**注意：** 为避免自动评分器（autograder）出错，你不能编辑或删除本实验中的非评分 cell，也请不要新增任何 cell。
**当你通过本作业后**，如果想对其中的非评分代码做实验，可以按 notebook 底部的说明操作。_

<a name="1"></a>
## 1 - 库 Packages

首先，运行下面的 cell 导入本作业需要的所有库。
- [numpy](www.numpy.org) 是 Python 中科学计算的基础库。
- [matplotlib](http://matplotlib.org) 是 Python 中著名的绘图库。
-  ``utils.py`` 包含本作业的辅助函数，你不需要修改这个文件里的代码。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
import copy
import math

%matplotlib inline

<a name="2"></a>
## 2 - 逻辑回归 Logistic Regression

在本练习的这一部分，你将建立一个逻辑回归模型，来预测某学生是否被大学录取。

<a name="2.1"></a>
### 2.1 问题描述

假设你是某大学院系的管理员，想根据每位申请者两门考试的成绩来判断其录取概率。
* 你有过往申请者的历史数据，可用作逻辑回归的训练集。
* 每个训练样本，你有申请者两门考试的成绩和录取决定。
* 你的任务是建立一个分类模型，根据这两门考试的成绩估计申请者的录取概率。

<a name="2.2"></a>
### 2.2 加载并可视化数据

你先加载本任务的数据集。
- 下面的 `load_dataset()` 函数把数据加载到变量 `X_train` 和 `y_train` 中
  - `X_train` 包含某学生两门考试的成绩
  - `y_train` 是录取决定
      - `y_train = 1` 表示该学生被录取
      - `y_train = 0` 表示该学生未被录取
  - `X_train` 和 `y_train` 都是 numpy 数组。

In [ ]:
# load dataset
X_train, y_train = load_data("data/ex2data1.txt")

#### 查看变量
我们更熟悉一下数据集。
- 一个好的起点就是把每个变量打印出来，看看它包含什么。

下面的代码打印 `X_train` 的前五个值以及该变量的类型。

In [ ]:
print("First five elements in X_train are:\n", X_train[:5])
print("Type of X_train:",type(X_train))

现在打印 `y_train` 的前五个值

In [ ]:
print("First five elements in y_train are:\n", y_train[:5])
print("Type of y_train:",type(y_train))

#### 查看变量的维度

熟悉数据的另一个有用方式是查看它的维度。我们打印 `X_train` 和 `y_train` 的 shape，看看数据集里有多少个训练样本。

In [ ]:
print ('The shape of X_train is: ' + str(X_train.shape))
print ('The shape of y_train is: ' + str(y_train.shape))
print ('We have m = %d training examples' % (len(y_train)))

#### 可视化你的数据

在开始实现任何学习算法之前，只要可能，可视化数据总是好的。
- 下面的代码把数据画在一张二维图上（如下所示），坐标轴是两门考试的成绩，正类和负类样本用不同的标记表示。
- 我们用 ``utils.py`` 文件中的一个辅助函数来生成这张图。

<img src="images/figure 1.png" width="450" height="450">

In [ ]:
# Plot examples
plot_data(X_train, y_train[:], pos_label="Admitted", neg_label="Not admitted")

# Set the y-axis label
plt.ylabel('Exam 2 score') 
# Set the x-axis label
plt.xlabel('Exam 1 score') 
plt.legend(loc="upper right")
plt.show()

你的目标是建立一个逻辑回归模型来拟合这份数据。
- 有了这个模型，你就能根据一个新学生两门考试的成绩来预测他是否会被录取。

<a name="2.3"></a>
### 2.3  Sigmoid 函数

回忆一下，逻辑回归的模型表示为

$$ f_{\mathbf{w},b}(x) = g(\mathbf{w}\cdot \mathbf{x} + b)$$
其中函数 $g$ 是 sigmoid 函数，定义为：

$$g(z) = \frac{1}{1+e^{-z}}$$

我们先实现 sigmoid 函数，好让本作业其余部分能用它。

<a name='ex-01'></a>
### 练习 Exercise 1
请完成 `sigmoid` 函数来计算

$$g(z) = \frac{1}{1+e^{-z}}$$

注意
- `z` 不总是单个数，也可以是一个数字数组。
- 如果输入是数字数组，我们希望对输入数组中的每个值都施加 sigmoid 函数。

如果卡住了，可以查看下面 cell 之后给出的提示来帮助实现。

In [ ]:
# UNQ_C1
# GRADED FUNCTION: sigmoid

def sigmoid(z):
    """
    Compute the sigmoid of z

    Args:
        z (ndarray): A scalar, numpy array of any size.

    Returns:
        g (ndarray): sigmoid(z), with the same shape as z
         
    """
          
    ### START CODE HERE ### 
    g = 0.0
    g = 1 / (1 + np.exp(-z))
    ### END SOLUTION ###  
    
    return g

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>

   * `numpy` 有一个函数 [`np.exp()`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html)，能方便地计算输入数组（`z`）中所有元素的指数（$e^{z}$）。

<details>
          <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>

  - 你可以把 $e^{-z}$ 写成代码 `np.exp(-z)`

  - 你可以把 $1/e^{-z}$ 写成代码 `1/np.exp(-z)`

    如果还是卡住，可以看下面的提示来弄清怎么计算 `g`

    <details>
          <summary><font size="2" color="darkblue"><b>计算 g 的提示</b></font></summary>
        <code>g = 1 / (1 + np.exp(-z))</code>
    </details>


</details>

完成后，试着在下面的 cell 中调用 `sigmoid(x)` 测试几个值。
- 对很大的正 x，sigmoid 应接近 1；对很大的负 x，sigmoid 应接近 0。
- `sigmoid(0)` 应恰好等于 0.5。

In [ ]:
# Note: You can edit this value
value = 0

print (f"sigmoid({value}) = {sigmoid(value)}")

**期望输出**：
<table>
  <tr>
    <td> <b>sigmoid(0)<b></td>
    <td> 0.5 </td>
  </tr>
</table>

- 如前所述，你的代码也应能处理向量和矩阵。对矩阵，你的函数应对每个元素施加 sigmoid 函数。

In [ ]:
print ("sigmoid([ -1, 0, 1, 2]) = " + str(sigmoid(np.array([-1, 0, 1, 2]))))

# UNIT TESTS
from public_tests import *
sigmoid_test(sigmoid)

**期望输出**：
<table>
  <tr>
    <td><b>sigmoid([-1, 0, 1, 2])<b></td>
    <td>[0.26894142        0.5           0.73105858        0.88079708]</td>
  </tr>

</table>

<a name="2.4"></a>
### 2.4 逻辑回归的代价函数

本节你将实现逻辑回归的代价函数。

<a name='ex-02'></a>
### 练习 Exercise 2

请用下面的方程完成 `compute_cost` 函数。

回忆一下，逻辑回归的代价函数形如

$$ J(\mathbf{w},b) = \frac{1}{m}\sum_{i=0}^{m-1} \left[ loss(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)}) \right] \tag{1}$$

其中
* m 是数据集中的训练样本数量

* $loss(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)})$ 是单个数据点的代价，即——

    $$loss(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)}) = (-y^{(i)} \log\left(f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) - \left( 1 - y^{(i)}\right) \log \left( 1 - f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) \tag{2}$$

*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是模型的预测值，$y^{(i)}$ 是真实标签

*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = g(\mathbf{w} \cdot \mathbf{x^{(i)}} + b)$，其中函数 $g$ 是 sigmoid 函数。
    * 在计算 $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = g(z_{\mathbf{w},b}(\mathbf{x}^{(i)}))$ 之前，先算一个中间变量 $z_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x^{(i)}} + b = w_0x^{(i)}_0 + ... + w_{n-1}x^{(i)}_{n-1} + b$（$n$ 是特征数量）会有帮助

注意：
* 做这个时记住，变量 `X_train` 和 `y_train` 不是标量，而是形状分别为 ($m, n$) 和 ($𝑚$,1) 的矩阵，其中 $𝑛$ 是特征数量，$𝑚$ 是训练样本数量。
* 这一部分你可以用上面实现的 sigmoid 函数。

如果卡住了，可以查看下面 cell 之后给出的提示来帮助实现。

In [ ]:
# UNQ_C2
# GRADED FUNCTION: compute_cost
def compute_cost(X, y, w, b, *argv):
    """
    Computes the cost over all examples
    Args:
      X : (ndarray Shape (m,n)) data, m examples by n features
      y : (ndarray Shape (m,))  target value 
      w : (ndarray Shape (n,))  values of parameters of the model      
      b : (scalar)              value of bias parameter of the model
      *argv : unused, for compatibility with regularized version below
    Returns:
      total_cost : (scalar) cost 
    """

    m, n = X.shape
    
    ### START CODE HERE ###        
    total_cost = 0.0
    for i in range(m):
        z = np.dot(w,X[i]) +b
        f_wb_i = sigmoid(z)
        loss_i = -y[i] * np.log(f_wb_i) - (1 - y[i]) * np.log(1 - f_wb_i)
        total_cost += loss_i
    total_cost /= m
    ### END CODE HERE ### 

    return total_cost

<details>
<summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>

* 求和算子（如 $h = \sum\limits_{i = 0}^{m-1} 2i$）可以在代码中这样表示：

```python
    h = 0
    for i in range(m):
        h = h + 2*i
```
<br>

* 本例中，你可以用 for 循环遍历 `X` 中所有样本，把每次迭代的 `loss` 累加到一个在循环外初始化的变量（`loss_sum`）上。

* 然后，把 `total_cost` 作为 `loss_sum` 除以 `m` 返回。

* 如果你刚接触 Python，请检查代码缩进是否用了一致的空格或 tab，否则可能产生不同的输出、或报 `IndentationError: unexpected indent` 错误。详情可参考社区里的[这个话题](https://community.deeplearning.ai/t/indentation-in-python-indentationerror-unexpected-indent/159398)。

<details>
<summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>

* 这个函数整体实现可以这样组织

```python
def compute_cost(X, y, w, b, *argv):
    m, n = X.shape

    ### START CODE HERE ###
    loss_sum = 0

    # Loop over each training example
    for i in range(m):

        # First calculate z_wb = w[0]*X[i][0]+...+w[n-1]*X[i][n-1]+b
        z_wb = 0
        # Loop over each feature
        for j in range(n):
            # Add the corresponding term to z_wb
            z_wb_ij = # Your code here to calculate w[j] * X[i][j]
            z_wb += z_wb_ij # equivalent to z_wb = z_wb + z_wb_ij
        # Add the bias term to z_wb
        z_wb += b # equivalent to z_wb = z_wb + b

        f_wb = # Your code here to calculate prediction f_wb for a training example
        loss =  # Your code here to calculate loss for a training example

        loss_sum += loss # equivalent to loss_sum = loss_sum + loss

    total_cost = (1 / m) * loss_sum
    ### END CODE HERE ###

    return total_cost
```
<br>

如果还是卡住，可以看下面的提示来弄清怎么计算 `z_wb_ij`、`f_wb` 和 `cost`。

<details>
<summary><font size="2" color="darkblue"><b>计算 z_wb_ij 的提示</b></font></summary>
           &emsp; &emsp; <code>z_wb_ij = w[j]*X[i][j] </code>
</details>

<details>
          <summary><font size="2" color="darkblue"><b>计算 f_wb 的提示</b></font></summary>
           &emsp; &emsp; $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = g(z_{\mathbf{w},b}(\mathbf{x}^{(i)}))$，其中 $g$ 是 sigmoid 函数。你可以直接调用上面实现的 `sigmoid` 函数。
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 f 的更多提示</b></font></summary>
               &emsp; &emsp; 你可以把 f_wb 算作 <code>f_wb = sigmoid(z_wb) </code>
           </details>
</details>

<details>
          <summary><font size="2" color="darkblue"><b>计算 loss 的提示</b></font></summary>
          &emsp; &emsp; 你可以用 <a href="https://numpy.org/doc/stable/reference/generated/numpy.log.html">np.log</a> 函数来计算对数
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 loss 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以把 loss 算作 <code>loss =  -y[i] * np.log(f_wb) - (1 - y[i]) * np.log(1 - f_wb)</code>
</details>
</details>

</details>

</details>

运行下面的 cell，用两组不同的 $w$、$b$ 初值来检查你对 `compute_cost` 函数的实现

In [ ]:
m, n = X_train.shape

# Compute and display cost with w and b initialized to zeros
initial_w = np.zeros(n)
initial_b = 0.
cost = compute_cost(X_train, y_train, initial_w, initial_b)
print('Cost at initial w and b (zeros): {:.3f}'.format(cost))

**期望输出**：
<table>
  <tr>
    <td> <b>初始 w、b (zeros) 处的代价<b></td>
    <td> 0.693 </td>
  </tr>
</table>

In [ ]:
# Compute and display cost with non-zero w and b
test_w = np.array([0.2, 0.2])
test_b = -24.
cost = compute_cost(X_train, y_train, test_w, test_b)

print('Cost at test w and b (non-zeros): {:.3f}'.format(cost))


# UNIT TESTS
compute_cost_test(compute_cost)

**期望输出**：
<table>
  <tr>
    <td> <b>测试 w、b (non-zeros) 处的代价：<b></td>
    <td> 0.218 </td>
  </tr>
</table>

<a name="2.5"></a>
### 2.5 逻辑回归的梯度

本节你将实现逻辑回归的梯度。

回忆一下，梯度下降算法为：

$$\begin{align*}& \text{repeat until convergence:} \; \lbrace \newline \; & b := b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b} \newline       \; & w_j := w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{for j := 0..n-1}\newline & \rbrace\end{align*}$$

其中参数 $b$、$w_j$ 全部同时更新


<a name='ex-03'></a>
### 练习 Exercise 3

请完成 `compute_gradient` 函数，用下面的式 (2)、(3) 计算 $\frac{\partial J(\mathbf{w},b)}{\partial w}$、$\frac{\partial J(\mathbf{w},b)}{\partial b}$。

$$
\frac{\partial J(\mathbf{w},b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)}) \tag{2}
$$
$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)})x_{j}^{(i)} \tag{3}
$$
* m 是数据集中的训练样本数量

*  $f_{\mathbf{w},b}(x^{(i)})$ 是模型的预测值，$y^{(i)}$ 是真实标签

- **注意**：虽然这个梯度看起来和线性回归的梯度一模一样，但公式其实不同，因为线性回归和逻辑回归对 $f_{\mathbf{w},b}(x)$ 的定义不同。

和之前一样，你可以用上面实现的 sigmoid 函数；如果卡住了，可以查看下面 cell 之后给出的提示来帮助实现。

In [ ]:
# UNQ_C3
# GRADED FUNCTION: compute_gradient
def compute_gradient(X, y, w, b, *argv): 
    """
    Computes the gradient for logistic regression 
 
    Args:
      X : (ndarray Shape (m,n)) data, m examples by n features
      y : (ndarray Shape (m,))  target value 
      w : (ndarray Shape (n,))  values of parameters of the model      
      b : (scalar)              value of bias parameter of the model
      *argv : unused, for compatibility with regularized version below
    Returns
      dj_dw : (ndarray Shape (n,)) The gradient of the cost w.r.t. the parameters w. 
      dj_db : (scalar)             The gradient of the cost w.r.t. the parameter b. 
    """
    m, n = X.shape
    dj_dw = np.zeros(w.shape)
    dj_db = 0.0

    ### START CODE HERE ### 
    for i in range(m):
        z_i = np.dot(X[i],w) + b
        f_wb_i = sigmoid(z_i)
        err_i = f_wb_i - y[i]
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err_i * X[i,j]
        dj_db = dj_db + err_i
    dj_db /= m
    dj_dw /= m
        
    ### END CODE HERE ###

        
    return dj_db, dj_dw

 <details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>


* 这个函数整体实现可以这样组织
    ```python
       def compute_gradient(X, y, w, b, *argv):
            m, n = X.shape
            dj_dw = np.zeros(w.shape)
            dj_db = 0.

            ### START CODE HERE ###
            for i in range(m):
                # Calculate f_wb (exactly as you did in the compute_cost function above)
                f_wb =

                # Calculate the  gradient for b from this example
                dj_db_i = # Your code here to calculate the error

                # add that to dj_db
                dj_db += dj_db_i

                # get dj_dw for each attribute
                for j in range(n):
                    # You code here to calculate the gradient from the i-th example for j-th attribute
                    dj_dw_ij =
                    dj_dw[j] += dj_dw_ij

            # divide dj_db and dj_dw by total number of examples
            dj_dw = dj_dw / m
            dj_db = dj_db / m
            ### END CODE HERE ###

            return dj_db, dj_dw
    ```

    * 如果你刚接触 Python，请检查代码缩进是否一致，否则可能报 `IndentationError: unexpected indent`。详情见社区[这个话题](https://community.deeplearning.ai/t/indentation-in-python-indentationerror-unexpected-indent/159398)。
    * 如果还是卡住，可以看下面的提示来弄清怎么计算 `f_wb`、`dj_db_i` 和 `dj_dw_ij`

    <details>
          <summary><font size="2" color="darkblue"><b>计算 f_wb 的提示</b></font></summary>
           &emsp; &emsp; 回忆你在上面 <code>compute_cost</code> 里算过 f_wb —— 各中间项的详细提示见那个练习下方的提示区
           <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 f_wb 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以这样算 f_wb
               <pre>
               for i in range(m):
                   # Calculate f_wb (exactly how you did it in the compute_cost function above)
                   z_wb = 0
                   # Loop over each feature
                   for j in range(n):
                       # Add the corresponding term to z_wb
                       z_wb_ij = X[i, j] * w[j]
                       z_wb += z_wb_ij

                   # Add bias term
                   z_wb += b

                   # Calculate the prediction from the model
                   f_wb = sigmoid(z_wb)
    </details>

    </details>
    <details>
          <summary><font size="2" color="darkblue"><b>计算 dj_db_i 的提示</b></font></summary>
           &emsp; &emsp; 你可以把 dj_db_i 算作 <code>dj_db_i = f_wb - y[i]</code>
    </details>

    <details>
          <summary><font size="2" color="darkblue"><b>计算 dj_dw_ij 的提示</b></font></summary>
        &emsp; &emsp; 你可以把 dj_dw_ij 算作 <code>dj_dw_ij = (f_wb - y[i])* X[i][j]</code>
    </details>

</details>

运行下面的 cell，用两组不同的 $w$、$b$ 初值来检查你对 `compute_gradient` 函数的实现

In [ ]:
# Compute and display gradient with w and b initialized to zeros
initial_w = np.zeros(n)
initial_b = 0.

dj_db, dj_dw = compute_gradient(X_train, y_train, initial_w, initial_b)
print(f'dj_db at initial w and b (zeros):{dj_db}' )
print(f'dj_dw at initial w and b (zeros):{dj_dw.tolist()}' )

**期望输出**：
<table>
  <tr>
    <td> <b>初始 w、b (zeros) 处的 dj_db<b></td>
    <td> -0.1 </td>
  </tr>
  <tr>
    <td> <b>初始 w、b (zeros) 处的 dj_dw：<b></td>
    <td> [-12.00921658929115, -11.262842205513591] </td>
  </tr>
</table>

In [ ]:
# Compute and display cost and gradient with non-zero w and b
test_w = np.array([ 0.2, -0.5])
test_b = -24
dj_db, dj_dw  = compute_gradient(X_train, y_train, test_w, test_b)

print('dj_db at test w and b:', dj_db)
print('dj_dw at test w and b:', dj_dw.tolist())

# UNIT TESTS    
compute_gradient_test(compute_gradient)

**期望输出**：
<table>
  <tr>
    <td> <b>测试 w、b (non-zeros) 处的 dj_db<b></td>
    <td> -0.5999999999991071 </td>
  </tr>
  <tr>
    <td> <b>测试 w、b (non-zeros) 处的 dj_dw：<b></td>
    <td>  [-44.8313536178737957, -44.37384124953978] </td>
  </tr>
</table>

<a name="2.6"></a>
### 2.6 用梯度下降学习参数

与上一个作业类似，现在你将用梯度下降找到逻辑回归模型的最优参数。
- 这部分你不需要实现任何东西，直接运行下面的 cell 即可。

- 验证梯度下降是否正确工作的一个好方法，是查看 $J(\mathbf{w},b)$ 的值、检查它是否每一步都在下降。

- 假如你正确实现了梯度、正确计算了代价，那么 $J(\mathbf{w},b)$ 应当从不上升，并在算法结束时收敛到一个稳定值。

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters, lambda_): 
    """
    Performs batch gradient descent to learn theta. Updates theta by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      X :    (ndarray Shape (m, n) data, m examples by n features
      y :    (ndarray Shape (m,))  target value 
      w_in : (ndarray Shape (n,))  Initial values of parameters of the model
      b_in : (scalar)              Initial value of parameter of the model
      cost_function :              function to compute cost
      gradient_function :          function to compute gradient
      alpha : (float)              Learning rate
      num_iters : (int)            number of iterations to run gradient descent
      lambda_ : (scalar, float)    regularization constant
      
    Returns:
      w : (ndarray Shape (n,)) Updated values of parameters of the model after
          running gradient descent
      b : (scalar)                Updated value of parameter of the model after
          running gradient descent
    """
    
    # number of training examples
    m = len(X)
    
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    w_history = []
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_db, dj_dw = gradient_function(X, y, w_in, b_in, lambda_)   

        # Update Parameters using w, b, alpha and gradient
        w_in = w_in - alpha * dj_dw               
        b_in = b_in - alpha * dj_db              
       
        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            cost =  cost_function(X, y, w_in, b_in, lambda_)
            J_history.append(cost)

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters/10) == 0 or i == (num_iters-1):
            w_history.append(w_in)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}   ")
        
    return w_in, b_in, J_history, w_history #return w and J,w history for graphing

现在我们运行上面的梯度下降算法，为数据集学习参数。

**注意**
下面的代码块要跑几分钟，尤其是非向量化的版本。你可以减小 `iterations` 来测试实现、加快迭代。以后有时间，可以跑 100,000 次迭代以获得更好的结果。

In [ ]:
np.random.seed(1)
initial_w = 0.01 * (np.random.rand(2) - 0.5)
initial_b = -8

# Some gradient descent settings
iterations = 10000
alpha = 0.001

w,b, J_history,_ = gradient_descent(X_train ,y_train, initial_w, initial_b, 
                                   compute_cost, compute_gradient, alpha, iterations, 0)

<details>
<summary>
    <b>期望输出：Cost 约 0.30，（点击查看细节）：</b>
</summary>

    # With the following settings
    np.random.seed(1)
    initial_w = 0.01 * (np.random.rand(2) - 0.5)
    initial_b = -8
    iterations = 10000
    alpha = 0.001
    #

```
Iteration    0: Cost     0.96
Iteration 1000: Cost     0.31
Iteration 2000: Cost     0.30
Iteration 3000: Cost     0.30
Iteration 4000: Cost     0.30
Iteration 5000: Cost     0.30
Iteration 6000: Cost     0.30
Iteration 7000: Cost     0.30
Iteration 8000: Cost     0.30
Iteration 9000: Cost     0.30
Iteration 9999: Cost     0.30
```

<a name="2.7"></a>
### 2.7 画出决策边界

现在我们用梯度下降得到的最终参数来画出线性拟合。如果你前面几部分实现正确，应当能看到类似下面这张的图：
<img src="images/figure 2.png"  width="450" height="450">

我们用 `utils.py` 文件中的一个辅助函数来生成这张图。

In [ ]:
plot_decision_boundary(w, b, X_train, y_train)
# Set the y-axis label
plt.ylabel('Exam 2 score') 
# Set the x-axis label
plt.xlabel('Exam 1 score') 
plt.legend(loc="upper right")
plt.show()

<a name="2.8"></a>
### 2.8 评估逻辑回归

我们可以通过查看"学到的模型在训练集上预测得有多好"来评估找到的参数的质量。

你将实现下面的 `predict` 函数来做这件事。

<a name='ex-04'></a>
### 练习 Exercise 4

请完成 `predict` 函数：给定一个数据集和学到的参数向量 $w$、$b$，产生 `1` 或 `0` 的预测。
- 首先你需要对每个样本计算模型的预测 $f(x^{(i)}) = g(w \cdot x^{(i)} + b)$
    - 你在上面几部分已经实现过
- 我们把模型的输出（$f(x^{(i)})$）解释为：在给定 $x^{(i)}$、由 $w$ 参数化下，$y^{(i)}=1$ 的概率。
- 因此，要从逻辑回归模型得到最终预测（$y^{(i)}=0$ 或 $y^{(i)}=1$），可以用如下判据——

  若 $f(x^{(i)}) >= 0.5$，预测 $y^{(i)}=1$

  若 $f(x^{(i)}) < 0.5$，预测 $y^{(i)}=0$

如果卡住了，可以查看下面 cell 之后给出的提示来帮助实现。

In [ ]:
# UNQ_C4
# GRADED FUNCTION: predict

def predict(X, w, b): 
    """
    Predict whether the label is 0 or 1 using learned logistic
    regression parameters w
    
    Args:
      X : (ndarray Shape (m,n)) data, m examples by n features
      w : (ndarray Shape (n,))  values of parameters of the model      
      b : (scalar)              value of bias parameter of the model

    Returns:
      p : (ndarray (m,)) The predictions for X using a threshold at 0.5
    """
    # number of training examples
    m, n = X.shape   
    p = np.zeros(m)
   
    ### START CODE HERE ### 
    # Loop over each example
    for i in range(m):   
        z_wb = np.dot(X[i],w) 
        # Loop over each feature
        for j in range(n): 
            # Add the corresponding term to z_wb
            z_wb += 0
        
        # Add bias term 
        z_wb += b
        
        # Calculate the prediction for this example
        f_wb = sigmoid(z_wb)

        # Apply the threshold
        p[i] = 1 if f_wb>0.5 else 0
        
    ### END CODE HERE ### 
    return p

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>


* 这个函数整体实现可以这样组织
    ```python
       def predict(X, w, b):
            # number of training examples
            m, n = X.shape
            p = np.zeros(m)

            ### START CODE HERE ###
            # Loop over each example
            for i in range(m):

                # Calculate f_wb (exactly how you did it in the compute_cost function above)
                # using a couple of lines of code
                f_wb =

                # Calculate the prediction for that training example
                p[i] = # Your code here to calculate the prediction based on f_wb

            ### END CODE HERE ###
            return p
    ```

    如果还是卡住，可以看下面的提示来弄清怎么计算 `f_wb` 和 `p[i]`

    <details>
          <summary><font size="2" color="darkblue"><b>计算 f_wb 的提示</b></font></summary>
           &emsp; &emsp; 回忆你在上面 <code>compute_cost</code> 里算过 f_wb —— 各中间项的详细提示见那个练习下方的提示区
           <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 f_wb 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以这样算 f_wb
               <pre>
               for i in range(m):
                   # Calculate f_wb (exactly how you did it in the compute_cost function above)
                   z_wb = 0
                   # Loop over each feature
                   for j in range(n):
                       # Add the corresponding term to z_wb
                       z_wb_ij = X[i, j] * w[j]
                       z_wb += z_wb_ij

                   # Add bias term
                   z_wb += b

                   # Calculate the prediction from the model
                   f_wb = sigmoid(z_wb)
    </details>

    </details>
    <details>
          <summary><font size="2" color="darkblue"><b>计算 p[i] 的提示</b></font></summary>
           &emsp; &emsp; 举例来说，如果你想表达"y 小于 3 时 x = 1，否则为 0"，代码可写作 <code>x = y < 3 </code>。现在对 p[i] 做同样的事：f_wb >= 0.5 时 p[i] = 1，否则为 0。
           <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 p[i] 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以把 p[i] 算作 <code>p[i] = f_wb >= 0.5</code>
          </details>
    </details>

</details>

完成 `predict` 函数后，运行下面的代码，通过计算分类器预测正确的样本比例，来报告它的训练准确率。

In [ ]:
# Test your predict code
np.random.seed(1)
tmp_w = np.random.randn(2)
tmp_b = 0.3    
tmp_X = np.random.randn(4, 2) - 0.5

tmp_p = predict(tmp_X, tmp_w, tmp_b)
print(f'Output of predict: shape {tmp_p.shape}, value {tmp_p}')

# UNIT TESTS        
predict_test(predict)

**期望输出**

<table>
  <tr>
    <td> <b>predict 的输出：shape (4,)，值 [0. 1. 1. 1.]<b></td>
  </tr>
</table>

现在我们用它来计算训练集上的准确率

In [ ]:
#Compute accuracy on our training set
p = predict(X_train, w,b)
print('Train Accuracy: %f'%(np.mean(p == y_train) * 100))

<table>
  <tr>
    <td> <b>训练准确率（约）：<b></td>
    <td> 92.00 </td>
  </tr>
</table>

<a name="3"></a>
## 3 - 正则化逻辑回归 Regularized Logistic Regression

在本练习的这一部分，你将实现正则化逻辑回归，来预测某芯片厂生产的微芯片是否通过质量保证（QA）。在 QA 中，每个微芯片要经过多项测试以确保功能正常。

<a name="3.1"></a>
### 3.1 问题描述

假设你是该工厂的产品经理，手上有一些微芯片在两项不同测试中的结果。
- 根据这两项测试，你想判断微芯片应被接受还是拒绝。
- 为帮你决策，你有一份过往微芯片测试结果的数据集，可据此建立逻辑回归模型。

<a name="3.2"></a>
### 3.2 加载并可视化数据

与本练习前面几部分类似，我们先加载本任务的数据集并把它可视化。

- 下面的 `load_dataset()` 函数把数据加载到变量 `X_train` 和 `y_train` 中
  - `X_train` 包含微芯片在两项测试中的结果
  - `y_train` 包含 QA 结果
      - `y_train = 1` 表示该微芯片被接受
      - `y_train = 0` 表示该微芯片被拒绝
  - `X_train` 和 `y_train` 都是 numpy 数组。

In [ ]:
# load dataset
X_train, y_train = load_data("data/ex2data2.txt")

#### 查看变量

下面的代码打印 `X_train` 和 `y_train` 的前五个值以及变量的类型。

In [ ]:
# print X_train
print("X_train:", X_train[:5])
print("Type of X_train:",type(X_train))

# print y_train
print("y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

#### 查看变量的维度

熟悉数据的另一个有用方式是查看它的维度。我们打印 `X_train` 和 `y_train` 的 shape，看看数据集里有多少个训练样本。

In [ ]:
print ('The shape of X_train is: ' + str(X_train.shape))
print ('The shape of y_train is: ' + str(y_train.shape))
print ('We have m = %d training examples' % (len(y_train)))

#### 可视化你的数据

辅助函数 `plot_data`（来自 `utils.py`）用来生成类似图 3 的图，坐标轴是两项测试的分数，正类（y = 1，接受）和负类（y = 0，拒绝）样本用不同的标记表示。

<img src="images/figure 3.png"  width="450" height="450">

In [ ]:
# Plot examples
plot_data(X_train, y_train[:], pos_label="Accepted", neg_label="Rejected")

# Set the y-axis label
plt.ylabel('Microchip Test 2') 
# Set the x-axis label
plt.xlabel('Microchip Test 1') 
plt.legend(loc="upper right")
plt.show()

图 3 表明，我们的数据集无法用一条穿过图的直线把正类和负类分开。因此，直接套用逻辑回归在这个数据集上效果不会好，因为逻辑回归只能找到线性的决策边界。

<a name="3.3"></a>
### 3.3 特征映射 Feature mapping

更好地拟合数据的一个办法，是从每个数据点创建更多特征。在给定的 `map_feature` 函数中，我们把特征映射为 $x_1$ 和 $x_2$ 直到六次方的所有多项式项。

$$\mathrm{map\_feature}(x) =
\left[\begin{array}{c}
x_1\\
x_2\\
x_1^2\\
x_1 x_2\\
x_2^2\\
x_1^3\\
\vdots\\
x_1 x_2^5\\
x_2^6\end{array}\right]$$

经过这个映射，我们原本两个特征（两项 QA 测试的分数）的向量被变换成了一个 27 维向量。

- 在这个更高维特征向量上训练的逻辑回归分类器会有更复杂的决策边界，画在二维图上时是非线性的。
- 我们已在 utils.py 中为你提供了 `map_feature` 函数。

In [ ]:
print("Original shape of data:", X_train.shape)

mapped_X =  map_feature(X_train[:, 0], X_train[:, 1])
print("Shape after feature mapping:", mapped_X.shape)

我们也打印 `X_train` 和 `mapped_X` 的前几个元素，看看这个变换。

In [ ]:
print("X_train[0]:", X_train[0])
print("mapped X_train[0]:", mapped_X[0])

特征映射虽然让我们能构建更有表达力的分类器，但也更容易过拟合。在本练习接下来的部分，你将实现正则化逻辑回归来拟合数据，并亲眼看看正则化如何帮助对抗过拟合问题。

<a name="3.4"></a>
### 3.4 正则化逻辑回归的代价函数

本部分你将实现正则化逻辑回归的代价函数。

回忆一下，正则化逻辑回归的代价函数形如
$$J(\mathbf{w},b) = \frac{1}{m}  \sum_{i=0}^{m-1} \left[ -y^{(i)} \log\left(f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) - \left( 1 - y^{(i)}\right) \log \left( 1 - f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) \right] + \frac{\lambda}{2m}  \sum_{j=0}^{n-1} w_j^2$$

把它与没有正则化的代价函数（你在上面实现过的）对比，后者形如

$$ J(\mathbf{w}.b) = \frac{1}{m}\sum_{i=0}^{m-1} \left[ (-y^{(i)} \log\left(f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) - \left( 1 - y^{(i)}\right) \log \left( 1 - f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right)\right]$$

差别就在于正则化项 $$\frac{\lambda}{2m}  \sum_{j=0}^{n-1} w_j^2$$
注意参数 $b$ 不做正则化。

<a name='ex-05'></a>
### 练习 Exercise 5

请完成下面的 `compute_cost_reg` 函数，对 $w$ 中每个元素计算下面这一项
$$\frac{\lambda}{2m}  \sum_{j=0}^{n-1} w_j^2$$

起始代码会把它加到"没有正则化的代价"（你在上面 `compute_cost` 里算过的）上，得到带正则化的代价。

如果卡住了，可以查看下面 cell 之后给出的提示来帮助实现。

In [ ]:
# UNQ_C5
def compute_cost_reg(X, y, w, b, lambda_ = 1):
    """
    Computes the cost over all examples
    Args:
      X : (ndarray Shape (m,n)) data, m examples by n features
      y : (ndarray Shape (m,))  target value 
      w : (ndarray Shape (n,))  values of parameters of the model      
      b : (scalar)              value of bias parameter of the model
      lambda_ : (scalar, float) Controls amount of regularization
    Returns:
      total_cost : (scalar)     cost 
    """

    m, n = X.shape
    
    # Calls the compute_cost function that you implemented above
    cost_without_reg = compute_cost(X, y, w, b) 
    
    # You need to calculate this value
    reg_cost = 0.0
    
    ### START CODE HERE ###
    reg_cost = sum(np.square(w))*lambda_/(2*m)  
    ### END CODE HERE ### 
    
    # Add the regularization cost to get the total cost
    total_cost = cost_without_reg + reg_cost

    return total_cost

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>


* 这个函数整体实现可以这样组织
    ```python
       def compute_cost_reg(X, y, w, b, lambda_ = 1):

           m, n = X.shape

            # Calls the compute_cost function that you implemented above
            cost_without_reg = compute_cost(X, y, w, b)

            # You need to calculate this value
            reg_cost = 0.

            ### START CODE HERE ###
            for j in range(n):
                reg_cost_j = # Your code here to calculate the cost from w[j]
                reg_cost = reg_cost + reg_cost_j
            reg_cost = (lambda_/(2 * m)) * reg_cost
            ### END CODE HERE ###

            # Add the regularization cost to get the total cost
            total_cost = cost_without_reg + reg_cost

        return total_cost
    ```

    如果还是卡住，可以看下面的提示来弄清怎么计算 `reg_cost_j`

    <details>
          <summary><font size="2" color="darkblue"><b>计算 reg_cost_j 的提示</b></font></summary>
           &emsp; &emsp; 你可以把 reg_cost_j 算作 <code>reg_cost_j = w[j]**2 </code>
    </details>

    </details>

</details>

运行下面的 cell 来检查你对 `compute_cost_reg` 函数的实现。

In [ ]:
X_mapped = map_feature(X_train[:, 0], X_train[:, 1])
np.random.seed(1)
initial_w = np.random.rand(X_mapped.shape[1]) - 0.5
initial_b = 0.5
lambda_ = 0.5
cost = compute_cost_reg(X_mapped, y_train, initial_w, initial_b, lambda_)

print("Regularized cost :", cost)

# UNIT TEST    
compute_cost_reg_test(compute_cost_reg)

**期望输出**：
<table>
  <tr>
    <td> <b>正则化代价 Regularized cost : <b></td>
    <td> 0.6618252552483948 </td>
  </tr>
</table>

<a name="3.5"></a>
### 3.5 正则化逻辑回归的梯度

本节你将实现正则化逻辑回归的梯度。

正则化代价函数的梯度有两部分。第一部分 $\frac{\partial J(\mathbf{w},b)}{\partial b}$ 是标量，另一部分是与参数 $\mathbf{w}$ 形状相同的向量，其第 $j$ 个元素定义如下：

$$\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m}  \sum_{i=0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})  $$

$$\frac{\partial J(\mathbf{w},b)}{\partial w_j} = \left( \frac{1}{m}  \sum_{i=0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) x_j^{(i)} \right) + \frac{\lambda}{m} w_j  \quad\, \mbox{for $j=0...(n-1)$}$$

把它与没有正则化的代价函数梯度（你在上面实现过的）对比，后者形如
$$
\frac{\partial J(\mathbf{w},b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)}) \tag{2}
$$
$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)})x_{j}^{(i)} \tag{3}
$$

可以看到，$\frac{\partial J(\mathbf{w},b)}{\partial b}$ 相同，差别在于 $\frac{\partial J(\mathbf{w},b)}{\partial w}$ 中多了下面这一项 $$\frac{\lambda}{m} w_j  \quad\, \mbox{for $j=0...(n-1)$}$$

<a name='ex-06'></a>
### 练习 Exercise 6

请完成下面的 `compute_gradient_reg` 函数，修改代码来计算下面这一项

$$\frac{\lambda}{m} w_j  \quad\, \mbox{for $j=0...(n-1)$}$$

起始代码会把它加到上面 `compute_gradient` 返回的 $\frac{\partial J(\mathbf{w},b)}{\partial w}$ 上，得到正则化代价函数的梯度。

如果卡住了，可以查看下面 cell 之后给出的提示来帮助实现。

In [ ]:
# UNQ_C6
def compute_gradient_reg(X, y, w, b, lambda_ = 1): 
    """
    Computes the gradient for logistic regression with regularization
 
    Args:
      X : (ndarray Shape (m,n)) data, m examples by n features
      y : (ndarray Shape (m,))  target value 
      w : (ndarray Shape (n,))  values of parameters of the model      
      b : (scalar)              value of bias parameter of the model
      lambda_ : (scalar,float)  regularization constant
    Returns
      dj_db : (scalar)             The gradient of the cost w.r.t. the parameter b. 
      dj_dw : (ndarray Shape (n,)) The gradient of the cost w.r.t. the parameters w. 

    """
    m, n = X.shape
    
    dj_db, dj_dw = compute_gradient(X, y, w, b)

    ### START CODE HERE ###     
    for j in range(n):
        dj_dw[j] = dj_dw[j] + (lambda_/m) * w[j]
    ### END CODE HERE ###          
        
    return dj_db, dj_dw

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>


* 这个函数整体实现可以这样组织
    ```python
    def compute_gradient_reg(X, y, w, b, lambda_ = 1):
        m, n = X.shape

        dj_db, dj_dw = compute_gradient(X, y, w, b)

        ### START CODE HERE ###
        # Loop over the elements of w
        for j in range(n):

            dj_dw_j_reg = # Your code here to calculate the regularization term for dj_dw[j]

            # Add the regularization term  to the correspoding element of dj_dw
            dj_dw[j] = dj_dw[j] + dj_dw_j_reg

        ### END CODE HERE ###

        return dj_db, dj_dw
    ```

    如果还是卡住，可以看下面的提示来弄清怎么计算 `dj_dw_j_reg`

    <details>
          <summary><font size="2" color="darkblue"><b>计算 dj_dw_j_reg 的提示</b></font></summary>
           &emsp; &emsp; 你可以把 dj_dw_j_reg 算作 <code>dj_dw_j_reg = (lambda_ / m) * w[j] </code>
    </details>

    </details>

</details>

运行下面的 cell 来检查你对 `compute_gradient_reg` 函数的实现。

In [ ]:
X_mapped = map_feature(X_train[:, 0], X_train[:, 1])
np.random.seed(1) 
initial_w  = np.random.rand(X_mapped.shape[1]) - 0.5 
initial_b = 0.5
 
lambda_ = 0.5
dj_db, dj_dw = compute_gradient_reg(X_mapped, y_train, initial_w, initial_b, lambda_)

print(f"dj_db: {dj_db}", )
print(f"First few elements of regularized dj_dw:\n {dj_dw[:4].tolist()}", )

# UNIT TESTS    
compute_gradient_reg_test(compute_gradient_reg)

**期望输出**：
<table>
  <tr>
    <td> <b>dj_db:</b>0.07138288792343</td> </tr>
  <tr>
      <td> <b> 正则化 dj_dw 的前几个元素：</b> </td> </tr>
   <tr>
   <td> [[-0.010386028450548], [0.011409852883280], [0.0536273463274], [0.003140278267313]] </td>
  </tr>
</table>

<a name="3.6"></a>
### 3.6 用梯度下降学习参数

与前面几部分类似，你将用上面实现的梯度下降函数来学习最优参数 $w$、$b$。
- 如果你正确完成了正则化逻辑回归的代价和梯度，就应该能一步步运行下面的 cell 来学习参数 $w$。
- 训练好参数后，我们会用它来画决策边界。

**注意**

下面的代码块要跑相当一段时间，尤其是非向量化的版本。你可以减小 `iterations` 来测试实现、加快迭代。以后有时间，跑 100,000 次迭代以获得更好的结果。

In [ ]:
# Initialize fitting parameters
np.random.seed(1)
initial_w = np.random.rand(X_mapped.shape[1])-0.5
initial_b = 1.

# Set regularization parameter lambda_ (you can try varying this)
lambda_ = 0.01    

# Some gradient descent settings
iterations = 100
alpha = 0.01

w,b, J_history,_ = gradient_descent(X_mapped, y_train, initial_w, initial_b, 
                                    compute_cost_reg, compute_gradient_reg, 
                                    alpha, iterations, lambda_)

<details>
<summary>
    <b>期望输出：Cost < 0.5 （点击查看细节）</b>
</summary>

```
# Using the following settings
#np.random.seed(1)
#initial_w = np.random.rand(X_mapped.shape[1])-0.5
#initial_b = 1.
#lambda_ = 0.01;
#iterations = 10000
#alpha = 0.01
Iteration    0: Cost     0.72
Iteration 1000: Cost     0.59
Iteration 2000: Cost     0.56
Iteration 3000: Cost     0.53
Iteration 4000: Cost     0.51
Iteration 5000: Cost     0.50
Iteration 6000: Cost     0.48
Iteration 7000: Cost     0.47
Iteration 8000: Cost     0.46
Iteration 9000: Cost     0.45
Iteration 9999: Cost     0.45

```

<a name="3.7"></a>
### 3.7 画出决策边界
为帮你可视化这个分类器学到的模型，我们会用 `plot_decision_boundary` 函数，它画出把正类和负类分开的（非线性）决策边界。

- 函数中，我们通过在一个均匀网格上计算分类器的预测、再画出"预测从 y = 0 变为 y = 1"处的等高线，来画这条非线性决策边界。

- 学到参数 $w$、$b$ 后，下一步就是画一条类似图 4 的决策边界。

<img src="images/figure 4.png"  width="450" height="450">

In [ ]:
plot_decision_boundary(w, b, X_mapped, y_train)
# Set the y-axis label
plt.ylabel('Microchip Test 2') 
# Set the x-axis label
plt.xlabel('Microchip Test 1') 
plt.legend(loc="upper right")
plt.show()

<a name="3.8"></a>
### 3.8 评估正则化逻辑回归模型

你将用上面实现的 `predict` 函数，计算正则化逻辑回归模型在训练集上的准确率

In [ ]:
#Compute accuracy on the training set
p = predict(X_mapped, w, b)

print('Train Accuracy: %f'%(np.mean(p == y_train) * 100))

**期望输出**：
<table>
  <tr>
    <td> <b>训练准确率 Train Accuracy:</b>~ 80%</td> </tr>
</table>

**恭喜你完成本课程的最后一个实验！我们希望在 Course 2 与你相见，你将在那里使用更高级的学习算法，如神经网络和决策树。继续学习！**

<details>
  <summary><font size="2" color="darkgreen"><b>如果你想对其中的非评分代码做实验，请点击这里。</b></font></summary>
    <p><i><b>重要提示：请务必在你已经通过本作业后再这样做，以免影响自动评分器。</b></i>
    <ol>
        <li> 在 notebook 菜单，点击 “View” > “Cell Toolbar” > “Edit Metadata”</li>
        <li> 点击你想锁定/解锁的 code cell 旁边的 “Edit Metadata” 按钮</li>
        <li> 把 “editable” 属性值设为：
            <ul>
                <li> “true” 表示解锁 </li>
                <li> “false” 表示锁定 </li>
            </ul>
        </li>
        <li> 在 notebook 菜单，点击 “View” > “Cell Toolbar” > “None” </li>
    </ol>
    <p> 下面是上述步骤的一个简短演示：
        <br>
        <img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" align="center" alt="unlock_cells.gif">
</details>